In [1]:
import pdfplumber
import csv
import re

In [2]:
column_headers = ['cip_year', 'project_type', 'source_page', 'department','project_name','project_id','start_year','end_year', 
                      'previous_appropriations', 'project_total']
years = {}
cip_year='2025-page141'
cleaned = []

In [7]:
def extract_field(text: str, label: str) -> str:
    next_labels = r'(?:Justification|Expenditure|Operating Budget|Relationship|Schedule|Summary)'
    m = re.search(label + r'\s*:\s*(.+?)(?=' + next_labels + r'|$)', text, re.S | re.I)
    if m:
        return re.sub(r'\s+', ' ', m.group(1)).strip()
    return ''

def clean_num(cell):
    if cell=='' or cell=='-':
        return 0
    cleaned_cell = cell.replace(",", "").replace(" ", "").replace("$", "")
    if cleaned_cell.startswith("(") and cleaned_cell.endswith(")"):
        cleaned_cell = "-" + cleaned_cell[1:-1]
    try:
        return int(cleaned_cell)
    except ValueError:
        return cleaned_cell

def extract_text_fields(txt, left_text, right_text):
    """Extract all free-text fields from page text."""
    lines = left_text.splitlines()
    department   = lines[0].strip() if lines else ''
    name_id      = re.match(r'(.+?)\s*/\s*([A-Z]\w+)', lines[1]) if len(lines) > 1 else None
    project_name = name_id.group(1).strip() if name_id else ''
    project_id   = name_id.group(2).strip() if name_id else ''
    project_type = right_text.splitlines()[0].strip() if right_text else ''

    cd_m  = re.search(r'Council District\s*:\s*(.+?)(?=Community|Priority|\n)', txt)
    cp_m  = re.search(r'Community Plan(?:ning)?\s*:\s*(.+?)(?=Project|Priority|\n)', txt)
    dur_m = re.search(r'Duration\s*:\s*(\d{4})\s*[-–]\s*(\d{4})', txt)

    address_location = '; '.join(filter(None, [
        'Council District: ' + cd_m.group(1).strip() if cd_m else '',
        'Community Plan: '   + cp_m.group(1).strip() if cp_m else '',
    ]))
    start_year = dur_m.group(1) if dur_m else ''
    end_year   = dur_m.group(2) if dur_m else ''

    description   = extract_field(left_text, 'Description')
    justification = extract_field(left_text, 'Justification')

    return {
        'department':        department,
        'project_name':      project_name,
        'project_id':        project_id,
        'project_type':      project_type,
        'address_location':  address_location,
        'start_year':        start_year,
        'end_year':          end_year,
        'description':       description,
        'justification':     justification,
    }


def extract_table_fields(pg_table):
    if not pg_table or len(pg_table) < 2:
        return {}, '', ''

    table = [
        [str(c or '').replace('\n', ' ').strip() for c in row]
        for row in pg_table
    ]

    total_row   = next((r for r in table if r[0].strip().lower() == 'total'), None)
    header_idx  = next((i for i, r in enumerate(table) if 'Fund Name' in r), None)

    if total_row is None or header_idx is None:
        return {}, '', ''

    # Merge the row above (partial labels) with the Fund Name row
    if header_idx > 0:
        above = table[header_idx - 1]
        header_row = [
            f"{above[i]} {table[header_idx][i]}".strip() if above[i] else table[header_idx][i]
            for i in range(len(table[header_idx]))
        ]
    else:
        header_row = table[header_idx]

    year_cols     = {}
    prev_approp   = ''
    project_total = ''

    for i, h in enumerate(header_row):
        if i >= len(total_row):
            break
        val = clean_num(total_row[i])

        if re.search(r'Exp', h, re.I):
            prev_approp = val
        elif re.search(r'Project\s*Total', h, re.I):
            project_total = val
        elif m := re.search(r'FY\s*(20\d{2})', h):
            yr  = m.group(1)
            key = f'year_{yr}_anticipated' if 'anticipated' in h.lower() else f'year_{yr}'
            year_cols[key] = val
        elif re.search(r'Future', h, re.I):
            year_cols['future_cost'] = val
        # Fund Name, Fund No, Con Appn, Unidentified Funding → skipped

    return year_cols, prev_approp, project_total


# ── main loop ────────────────────────────────────────────────────────────────

with pdfplumber.open(r"C:\Users\vince\Documents\GitHub\CIPBD\San-Diego\PDF\\" + f"{cip_year}.pdf") as pdf:
    for pg in pdf.pages:
        txt      = pg.extract_text() or ''
        pg_table = pg.extract_table()

        print(pg_table)

        if not (pg_table and "Duration" in txt and "Contact" in txt and "Justification" in txt):
            continue

        mid_x      = pg.width / 2
        left_text  = pg.within_bbox((0,     0, mid_x,    pg.height)).extract_text() or ''
        right_text = pg.within_bbox((mid_x, 0, pg.width, pg.height)).extract_text() or ''

        text_fields  = extract_text_fields(txt, left_text, right_text)
        year_cols, prev_approp, project_total = extract_table_fields(pg_table)

        row = {
            'cip_year':                cip_year,
            'project_type':            text_fields['project_type'],
            'source_page':             pg.page_number,
            'department':              text_fields['department'],
            'project_name':            text_fields['project_name'],
            'project_id':              text_fields['project_id'],
            'address_location':        text_fields['address_location'],
            'start_year':              text_fields['start_year'],
            'end_year':                text_fields['end_year'],
            'project_description':     text_fields['description'],
            'project_justification':   text_fields['justification'],
            'previous_appropriations': prev_approp,
            'project_total':           project_total,
        }
        row.update(year_cols)
        cleaned.append(row)

for key, value in cleaned[0].items():
    print(f"{key}: {value}")

[['', '', '', '', '', 'FY 2026', '', '', '', '', '', 'Unidentified', 'Project'], ['Fund Name', 'Fund No', 'Exp/Enc', 'Con Appn', 'FY 2026', 'Anticipated', 'FY 2027', 'FY 2028', 'FY 2029', 'FY 2030', 'Future FY', 'Funding', 'Total'], ['CIP Contributions from General Fund\nCitywide Library DIF\nCrossroads Redevelopmen CIP Contributions Fund\nDebt Funded General Fund CIP Projects\nGrant Fund - State\nLibrary Improvement Trust Fund\nLibrary System Improvement Fund\nUnidentified Funding', '400265\n400887\n200357\n400881\n600001\n200369\n200209\n9999', '$ 92,874 $ -\n- -\n429,517 32,366\n- 325,783\n4,930,338 15,369,662\n135,000 231,186\n- 699,468\n- -', None, '$ -', '$ - $ - $ - $ - $ - $ - $ - $ 92,874\n- - - - - - - 1,000,000\n- - - - - - - 461,883\n- - - - - - - 5,273,675\n9,044,317 - - - - - - 29,344,317\n- - - - - - - 366,186\n- - - - - - - 699,468\n- - - - - - 46,596 46,596', None, None, None, None, None, None, None], [None, None, None, None, '1,000,000', None, None, None, None, None, 

In [10]:
raw = [['', '', '', '', '', 'FY 2026', '', '', '', '', '', 'Unidentified', 'Project'], ['Fund Name', 'Fund No', 'Exp/Enc', 'Con Appn', 'FY 2026', 'Anticipated', 'FY 2027', 'FY 2028', 'FY 2029', 'FY 2030', 'Future FY', 'Funding', 'Total'], ['CIP Contributions from General Fund\nCitywide Library DIF\nCrossroads Redevelopmen CIP Contributions Fund\nDebt Funded General Fund CIP Projects\nGrant Fund - State\nLibrary Improvement Trust Fund\nLibrary System Improvement Fund\nUnidentified Funding', '400265\n400887\n200357\n400881\n600001\n200369\n200209\n9999', '$ 92,874 $ -\n- -\n429,517 32,366\n- 325,783\n4,930,338 15,369,662\n135,000 231,186\n- 699,468\n- -', None, '$ -', '$ - $ - $ - $ - $ - $ - $ - $ 92,874\n- - - - - - - 1,000,000\n- - - - - - - 461,883\n- - - - - - - 5,273,675\n9,044,317 - - - - - - 29,344,317\n- - - - - - - 366,186\n- - - - - - - 699,468\n- - - - - - 46,596 46,596', None, None, None, None, None, None, None], [None, None, None, None, '1,000,000', None, None, None, None, None, None, None, None], [None, None, None, None, '-', None, None, None, None, None, None, None, None], [None, None, None, None, '4,947,892', None, None, None, None, None, None, None, None], [None, None, None, None, '-', None, None, None, None, None, None, None, None], [None, None, None, None, '-', None, None, None, None, None, None, None, None], [None, None, None, None, '-', None, None, None, None, None, None, None, None], [None, None, None, None, '-', None, None, None, None, None, None, None, None], ['Total', '', '$ 5,587,728', '$ 16,658,466', '$ 5,947,892', '$ 9,044,317', '$ -', '$ -', '$ -', '$ -', '$ -', '$ 46,596', '$ 37,285,000']]

# Build combined header from rows 0 and 1
header = [
    f"{(raw[0][i] or '').strip()} {(raw[1][i] or '').strip()}".strip()
    for i in range(len(raw[1]))
]

# Fund names and numbers are \n-separated in row 2
fund_names = raw[2][0].split('\n')
fund_nos   = raw[2][1].split('\n')
n_funds    = len(fund_names)

# Exp/Enc and Con Appn are paired on each line of col 2
exp_con = [re.sub(r'\$\s*', '', line).split() for line in raw[2][2].split('\n')]

# FY 2026 values are split across rows 2-9 in col 4
fy2026_vals = [
    re.sub(r'\$\s*', '', v or '-').strip()
    for v in [raw[2][4]] + [row[4] for row in raw[3:3 + n_funds - 1]]
]

# Remaining FY columns are \n-separated in col 5 of row 2
remaining_cols = [re.sub(r'\$\s*', '', line).split() for line in raw[2][5].split('\n')]
# Each line: FY2026Ant, FY2027, FY2028, FY2029, FY2030, FutFY, UnidFund, ProjTotal

rows = []
for i in range(n_funds):
    ec       = exp_con[i]       if i < len(exp_con)       else ['-', '-']
    fy26     = fy2026_vals[i]   if i < len(fy2026_vals)   else '-'
    rem      = remaining_cols[i] if i < len(remaining_cols) else ['-'] * 8

    rows.append({
        'Fund Name':             fund_names[i],
        'Fund No':               fund_nos[i],
        'Exp/Enc':               ec[0] if len(ec) > 0 else '-',
        'Con Appn':              ec[1] if len(ec) > 1 else '-',
        'FY 2026':               fy26,
        'FY 2026 Anticipated':   rem[0] if len(rem) > 0 else '-',
        'FY 2027':               rem[1] if len(rem) > 1 else '-',
        'FY 2028':               rem[2] if len(rem) > 2 else '-',
        'FY 2029':               rem[3] if len(rem) > 3 else '-',
        'FY 2030':               rem[4] if len(rem) > 4 else '-',
        'Future FY':             rem[5] if len(rem) > 5 else '-',
        'Unidentified Funding':  rem[6] if len(rem) > 6 else '-',
        'Project Total':         rem[7] if len(rem) > 7 else '-',
    })

# Print as markdown
cols = list(rows[0].keys())
print('| ' + ' | '.join(cols) + ' |')
print('| ' + ' | '.join(['---'] * len(cols)) + ' |')
for r in rows:
    print('| ' + ' | '.join(str(r[c]) for c in cols) + ' |')

| Fund Name | Fund No | Exp/Enc | Con Appn | FY 2026 | FY 2026 Anticipated | FY 2027 | FY 2028 | FY 2029 | FY 2030 | Future FY | Unidentified Funding | Project Total |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| CIP Contributions from General Fund | 400265 | 92,874 | - | - | - | - | - | - | - | - | - | 92,874 |
| Citywide Library DIF | 400887 | - | - | 1,000,000 | - | - | - | - | - | - | - | 1,000,000 |
| Crossroads Redevelopmen CIP Contributions Fund | 200357 | 429,517 | 32,366 | - | - | - | - | - | - | - | - | 461,883 |
| Debt Funded General Fund CIP Projects | 400881 | - | 325,783 | 4,947,892 | - | - | - | - | - | - | - | 5,273,675 |
| Grant Fund - State | 600001 | 4,930,338 | 15,369,662 | - | 9,044,317 | - | - | - | - | - | - | 29,344,317 |
| Library Improvement Trust Fund | 200369 | 135,000 | 231,186 | - | - | - | - | - | - | - | - | 366,186 |
| Library System Improvement Fund | 200209 | - | 699,468 | - | - | - | - | - | - | - | - | 699,468 |


In [17]:
import re

raw = [['', '', '', '', '', 'FY 2026', '', '', '', '', '', 'Unidentified', 'Project'], 
       ['Fund Name', 'Fund No', 'Exp/Enc', 'Con Appn', 'FY 2026', 'Anticipated', 'FY 2027', 'FY 2028', 'FY 2029', 'FY 2030', 'Future FY', 'Funding', 'Total'], 
       ['CIP Contributions from General Fund\nCitywide Library DIF\nCrossroads Redevelopmen CIP Contributions Fund\nDebt Funded General Fund CIP Projects\nGrant Fund - State\nLibrary Improvement Trust Fund\nLibrary System Improvement Fund\nUnidentified Funding', '400265\n400887\n200357\n400881\n600001\n200369\n200209\n9999', '$ 92,874 $ -\n- -\n429,517 32,366\n- 325,783\n4,930,338 15,369,662\n135,000 231,186\n- 699,468\n- -', None, '$ -', '$ - $ - $ - $ - $ - $ - $ - $ 92,874\n- - - - - - - 1,000,000\n- - - - - - - 461,883\n- - - - - - - 5,273,675\n9,044,317 - - - - - - 29,344,317\n- - - - - - - 366,186\n- - - - - - - 699,468\n- - - - - - 46,596 46,596', None, None, None, None, None, None, None], [None, None, None, None, '1,000,000', None, None, None, None, None, None, None, None], [None, None, None, None, '-', None, None, None, None, None, None, None, None], [None, None, None, None, '4,947,892', None, None, None, None, None, None, None, None], [None, None, None, None, '-', None, None, None, None, None, None, None, None], [None, None, None, None, '-', None, None, None, None, None, None, None, None], [None, None, None, None, '-', None, None, None, None, None, None, None, None], [None, None, None, None, '-', None, None, None, None, None, None, None, None], ['Total', '', '$ 5,587,728', '$ 16,658,466', '$ 5,947,892', '$ 9,044,317', '$ -', '$ -', '$ -', '$ -', '$ -', '$ 46,596', '$ 37,285,000']]

def clean_num(v):
    v = str(v or '').replace(',', '').replace('$', '').strip()
    if v in ('', '-', '—'): return 0
    if v.startswith('(') and v.endswith(')'): return -int(v[1:-1])
    try: return int(float(v))
    except: return 0

def extract_table_fields(pg_table):
    if not pg_table or len(pg_table) < 2:
        return {}, 0, 0

    table = [[str(c or '').strip() for c in row] for row in pg_table]

    header_idx = next((i for i, r in enumerate(table) if 'Fund Name' in r), None)
    total_row  = next((r for r in table if r[0].lower() == 'total'), None)

    if header_idx is None or total_row is None:
        return {}, 0, 0

    # Merge split header
    if header_idx > 0:
        above = table[header_idx - 1]
        header = [
            f"{above[i]} {table[header_idx][i]}".strip() if above[i] else table[header_idx][i]
            for i in range(len(table[header_idx]))
        ]
    else:
        header = table[header_idx]

    year_cols     = {}
    prev_approp   = 0
    project_total = 0

    for i, h in enumerate(header):
        if i >= len(total_row):
            break
        val = clean_num(total_row[i])

        if re.search(r'Exp|Con\s*App', h, re.I):
            prev_approp += val
        elif re.search(r'Project\s*Total', h, re.I):
            project_total = val
        elif m := re.search(r'FY\s*(20\d{2})', h):
            year_cols[f'year_{m.group(1)}'] = year_cols.get(f'year_{m.group(1)}', 0) + val
        elif re.search(r'Future', h, re.I):
            year_cols['future_cost'] = year_cols.get('future_cost', 0) + val

    return year_cols, prev_approp, project_total


def write_csv(records, filepath):
    """
    Writes a list of project dicts to CSV.
    Year columns are discovered dynamically and sorted.
    """
    if not records:
        return

    fixed_cols = [
        'cip_year', 'project_type', 'source_page', 'department',
        'project_name', 'start_year', 'end_year', 'address_location',
        'previous_appropriations', 'project_total',
    ]
    year_cols = sorted({
        k for r in records for k in r
        if k.startswith('year_') or k == 'future_cost'
    })
    fieldnames = fixed_cols + year_cols

    with open(filepath, 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames, extrasaction='ignore')
        writer.writeheader()
        for r in records:
            # Fill missing year cols with 0
            for col in year_cols:
                r.setdefault(col, 0)
            writer.writerow(r)

records = []
cip_year = "2025-page141"
pdf_path = r"C:\Users\vince\Documents\GitHub\CIPBD\San-Diego\PDF\\" + f"{cip_year}.pdf"

with pdfplumber.open(pdf_path) as pdf:
    for pg in pdf.pages:
        txt      = pg.extract_text() or ''
        pg_table = pg.extract_table()
        if not (pg_table and "Duration" in txt and "Justification" in txt):
            continue

        mid_x      = pg.width / 2
        left_text  = pg.within_bbox((0, 0, mid_x, pg.height)).extract_text() or ''
        right_text = pg.within_bbox((mid_x, 0, pg.width, pg.height)).extract_text() or ''

        text_fields              = extract_text_fields(txt, left_text, right_text)
        text_fields['source_page'] = pg.page_number
        year_cols, prev_approp, project_total = extract_table_fields(pg_table)

        record = {
            'cip_year':                cip_year,
            'project_type':            text_fields.get('project_type', ''),
            'source_page':             pg.page_number,
            'department':              text_fields.get('department', ''),
            'project_name':            text_fields.get('project_name', ''),
            'start_year':              text_fields.get('start_year', ''),
            'end_year':                text_fields.get('end_year', ''),
            'address_location':        text_fields.get('address_location', ''),
            'previous_appropriations': prev_approp,
            'project_total':           project_total,
        }
        record.update(year_cols)
        records.append(record)

write_csv(records, r"C:\Users\vince\Documents\GitHub\CIPBD\Scripts\San-Diego\outputs\\" + f"{cip_year}.csv")